# Fixing top-bottom qpath error

In [7]:
import glob

from natsort import natsorted

In [12]:
fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/qu_path/*.qpdata')

In [13]:
fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/qu_path/rep2_Mouse_1_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/qu_path/rep2_Mouse_2_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_top_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/qu_path/frame_t_0.ets - CF405, CF488, CF561.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/qu_path/rep2_Mouse_4_top_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/qu_path/rep2_Mouse_4_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slic

In [14]:
natsorted([fn for fn in fns if ('top' in fn) or ('bot' in fn)])

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/qu_path/Mouse_6_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/qu_path/Mouse_6_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/qu_path/Mouse_7_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_7/qu_path/Mouse_7_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_9/qu_path/Mouse_9_max_proj_bot.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_9/qu_path/Mouse_9_max_proj_top.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mouse_3_bot_max_proj.qpdata',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/qu_path/rep2_Mous

In [16]:
fns_to_fix = [fn for fn in fns if ('top' in fn) or ('bot' in fn)]

In [17]:
len(fns_to_fix)

30

In [1]:
import json
from pathlib import Path

import numpy as np
from shapely.geometry import shape
from shapely.validation import make_valid
from tqdm.auto import tqdm

In [6]:
import pandas as pd
from tqdm import tqdm


def count_and_summarize_geojson(geojson_path: Path) -> pd.DataFrame:
    """
    Loads a GeoJSON file, counts the number of valid ROIs (geometries),
    and returns a DataFrame summary of their properties.

    Args:
        geojson_path: Path to the input .geojson file.

    Returns:
        A pandas DataFrame containing the area and is_valid status for each ROI.
    """

    # Define the exclusion threshold in square pixels
    SIZE_THRESHOLD_PX2 = 2000

    print(f"\n🧠 Loading and analyzing {geojson_path.name}")
    try:
        with open(geojson_path) as f:
            gj_data = json.load(f)
    except FileNotFoundError:
        print(f"🛑 Error: File not found at {geojson_path}")
        return pd.DataFrame()

    features = gj_data.get("features", [])
    geoms = []
    properties_list = []
    
    # Initialize counters
    total_features = len(features)
    skipped_large_rois_count = 0
    
    # Iterate through features with a tqdm progress bar
    print(f"Processing GeoJSON features (Exclusion threshold: >{SIZE_THRESHOLD_PX2} px²):")
    for feat in tqdm(features, total=total_features, desc="Validating ROIs"):
        geom = feat.get("geometry")
        properties = feat.get("properties", {})

        if geom:
            g = shape(geom)

            # Skip polygons with area > SIZE_THRESHOLD_PX2 (likely FOV contour)
            if g.area > SIZE_THRESHOLD_PX2:
                skipped_large_rois_count += 1
                continue

            # Fix invalids/self-intersections
            try:
                g = make_valid(g)
            except Exception:
                g = g.buffer(0)

            if not g.is_empty:
                geoms.append(g)
                # Store geometry properties and calculated metrics
                props = {
                    "area_px2": g.area,
                    "is_valid": g.is_valid,
                    "geojson_filename": geojson_path.name,
                }
                # Add QuPath properties if available
                props.update(properties)
                properties_list.append(props)

    print("\n--- Analysis Summary ---")
    print(f"Total features in GeoJSON: {total_features}")
    print(f"Count of ROIs skipped due to size (> {SIZE_THRESHOLD_PX2} px²): {skipped_large_rois_count}")
    
    if not geoms:
        print(f"⚠️ No valid ROIs remaining after size filter in {geojson_path.name}")
        return pd.DataFrame()
        
    # Drop the largest polygon (likely the FOV contour) if more than one exists
    # This catches the scenario where the FOV contour was below the SIZE_THRESHOLD_PX2 (unlikely but possible)
    if len(geoms) > 1:
        areas = [g["area_px2"] for g in properties_list]
        max_idx = int(np.argmax(areas))

        # Filter both the geometries and the properties list
        largest_area = areas[max_idx]
        properties_list = [p for i, p in enumerate(properties_list) if i != max_idx]
        geoms = [g for i, g in enumerate(geoms) if i != max_idx]

        # The largest remaining geometry was removed as a final FOV safety check
        if largest_area > SIZE_THRESHOLD_PX2:
             # If the removed largest polygon was above the initial threshold, this is redundant,
             # but we can print a note for clarity.
             pass
        else:
             print(f"Note: The largest remaining ROI (Area: {largest_area:.2f} px²) was removed as a secondary FOV check.")
    
    if not geoms:
        print("⚠️ Only potential FOV contour present after final check, skipping.")
        return pd.DataFrame()

    # Create and return the DataFrame summary
    df_summary = pd.DataFrame(properties_list)
    print(f"✅ Found {len(df_summary)} valid ROIs for summary.")
    return df_summary

# --- Execution ---

# Define the single GeoJSON file you want to process
# Replace this with the specific file path you are interested in
geojson_file_to_process = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_12/qu_path/rep2_Mouse_12_bot_max_proj.geojson')

# Generate the DataFrame
results_df = count_and_summarize_geojson(geojson_file_to_process)

# Display the resulting DataFrame
if not results_df.empty:
    print("\nROI Summary DataFrame (First 5 Rows):")
    print(results_df.head())
    print(f"\nTotal ROIs retained in DataFrame: {len(results_df)}")
else:
    print("\nNo DataFrame generated.")


🧠 Loading and analyzing rep2_Mouse_12_bot_max_proj.geojson
Processing GeoJSON features (Exclusion threshold: >2000 px²):


Validating ROIs: 100%|██████████████████████████████████████████████████████| 121092/121092 [00:11<00:00, 10668.66it/s]



--- Analysis Summary ---
Total features in GeoJSON: 121092
Count of ROIs skipped due to size (> 2000 px²): 3027
Note: The largest remaining ROI (Area: 1998.55 px²) was removed as a secondary FOV check.
✅ Found 118064 valid ROIs for summary.

ROI Summary DataFrame (First 5 Rows):
    area_px2  is_valid                    geojson_filename objectType  \
0  111.72460      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
1   34.22840      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
2   82.17700      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
3  180.46655      True  rep2_Mouse_12_bot_max_proj.geojson  detection   
4  116.38620      True  rep2_Mouse_12_bot_max_proj.geojson  detection   

                                        measurements  
0  {'Nucleus: Area': 3.0, 'Nucleus: Perimeter': 1...  
1  {'Nucleus: Area': 1.0, 'Nucleus: Perimeter': 4...  
2  {'Nucleus: Area': 2.3125, 'Nucleus: Perimeter'...  
3  {'Nucleus: Area': 4.9375, 'Nucleus: Perimeter'...  
4  {'